# nb_03b — Gold: `fact_workforce_event` with as-of key resolution

We attribute each pay-setting event to
the **pay grid in force on the event date**, then compute a **compa-ratio** we can
compare against *today's* re-benchmarked grid.

Resolved per event:
- `date_key` → `dim_date`
- `cost_center_key` → `dim_cost_center` (SCD1, direct)
- `worker_key` → `dim_worker` **as-of** `event_date` (SCD2 range join)
- `pay_band_key` → `dim_pay_band` **as-of** on (group, level, event_date) (SCD2 range join)

We precompute band bounds and compa-ratio at event so the model's DAX stays simple.

## Load the fact inputs

**Summary.** Loads the Silver events and the dimensions, aliasing the SCD2 validity columns so the as-of range joins that follow read clearly.

<details>
<summary>Line-by-line details</summary>

- `ev = spark.table("silver.workforce_event")` — the conformed events to load.
- `dcc` — cost-centre key lookup (SCD1, direct join on `cost_center_id`).
- `dw` — worker versions with `effective_from`/`effective_to` aliased to `w_from`/`w_to` for the as-of join.
- `dpb` — pay-band versions with group/level aliased to `pb_group`/`pb_level` plus band bounds and validity range.
- `print(...)` — report how many events will be loaded.

</details>

In [ ]:
from pyspark.sql import functions as F
ev  = spark.table("silver.workforce_event")
dcc = spark.table("gold.dim_cost_center").select("cost_center_key","cost_center_id")
dw  = spark.table("gold.dim_worker").select("worker_key","employee_id",
        F.col("effective_from").alias("w_from"), F.col("effective_to").alias("w_to"))
dpb = spark.table("gold.dim_pay_band").select("pay_band_key",
        F.col("classification_group").alias("pb_group"),
        F.col("classification_level").alias("pb_level"),
        "band_min","band_mid","band_max","effective_from","effective_to")
print(f"silver events to load: {ev.count():,}")

## Build the fact with as-of key resolution

**Summary.** Resolves each event's dimension keys — cost centre directly, worker and pay band **as-of** the event date via SCD2 range joins — then precomputes the base salary, bonus, compa-ratio, and below-band flag.

<details>
<summary>Line-by-line details</summary>

- `date_key` — the `yyyyMMdd` integer join to `dim_date`.
- `.join(dcc, "cost_center_id", "left")` — direct SCD1 cost-centre key.
- Worker range join: `employee_id` equal **and** `event_date` between `w_from` and `w_to` — picks the worker version in force at the event.
- Pay-band range join: group and level equal **and** `event_date` between the band's `effective_from`/`effective_to` — picks the grid in force at the event.
- `base_salary_cad` — the CAD amount only for compensation events (`Hire`, `Promotion`, `Step Increment`).
- `bonus_cad` — the CAD amount only for `Performance Pay`.
- `compa_ratio_at_event` — `base_salary_cad / band_mid` (rounded) when a base salary and positive midpoint exist.
- `below_band_at_event` — whether the base salary is below the band minimum.
- The `.select(...)` shapes the final columns (band bounds aliased `*_at_event`); `write ... saveAsTable("gold.fact_workforce_event")` and `print` persist and report the row count.

</details>

In [ ]:
COMP = ["Hire","Promotion","Step Increment"]

fact = (ev
    .withColumn("date_key", F.date_format("event_date","yyyyMMdd").cast("int"))
    .join(dcc, "cost_center_id", "left")
    # as-of worker (SCD2 range join)
    .join(dw, (ev.employee_id==dw.employee_id) &
              (ev.event_date>=dw.w_from) & (ev.event_date<=dw.w_to), "left")
    # as-of pay band (SCD2 range join on group+level+date)
    .join(dpb, (ev.classification_group==dpb.pb_group) &
               (ev.classification_level==dpb.pb_level) &
               (ev.event_date>=dpb.effective_from) &
               (ev.event_date<=dpb.effective_to), "left")
    # measures
    .withColumn("base_salary_cad",
        F.when(F.col("event_type").isin(COMP), F.col("amount_cad")))
    .withColumn("bonus_cad",
        F.when(F.col("event_type")=="Performance Pay", F.col("amount_cad")))
    .withColumn("compa_ratio_at_event",
        F.when(F.col("base_salary_cad").isNotNull() & (F.col("band_mid")>0),
               F.round(F.col("base_salary_cad")/F.col("band_mid"),4)))
    .withColumn("below_band_at_event",
        F.when(F.col("base_salary_cad").isNotNull(),
               F.col("base_salary_cad") < F.col("band_min")))
    .select("event_id","date_key","cost_center_key","worker_key","pay_band_key",
            "classification_group","classification_level","event_type",
            "amount_cad","base_salary_cad","bonus_cad",
            F.col("band_min").alias("band_min_at_event"),
            F.col("band_mid").alias("band_mid_at_event"),
            F.col("band_max").alias("band_max_at_event"),
            "compa_ratio_at_event","below_band_at_event",
            "local_currency","source_system","ingest_ts"))

(fact.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.fact_workforce_event"))

print(f"fact rows: {fact.count():,}")

## Data-quality gate — no unresolved keys

**Summary.** Counts any rows where a dimension key failed to resolve, so a broken join surfaces immediately instead of silently corrupting analysis.

<details>
<summary>Line-by-line details</summary>

- The `SELECT sum(case when ... is null ...)` query counts null `worker_key`, `pay_band_key`, and `cost_center_key` alongside the total row count. All null counts should be zero.

</details>

In [ ]:
spark.sql("""
  SELECT sum(case when worker_key    is null then 1 else 0 end) AS null_worker,
         sum(case when pay_band_key  is null then 1 else 0 end) AS null_pay_band,
         sum(case when cost_center_key is null then 1 else 0 end) AS null_cost_center,
         count(*) AS total
  FROM gold.fact_workforce_event""").show()

## Compa-ratio as-was vs as-is

Average compa-ratio of pay set in 2021, measured against the grid **in force then**
(as-was) — every such pay looks healthy near 1.0. But the grid has since been
re-benchmarked upward, so the same salaries sit lower against **today's** grid.


<details>
<summary>Line-by-line details</summary>

- The query joins the fact to `dim_date`, filters to rows with a `base_salary_cad`, and groups by `d.year` (the year pay was set).
- `avg(f.compa_ratio_at_event)` — the average as-was compa-ratio per year.
- `sum(case when f.below_band_at_event then 1 else 0 end)` — the as-was count of below-band pay per year.

</details>

In [ ]:
spark.sql("""
  SELECT d.year AS pay_set_year,
         round(avg(f.compa_ratio_at_event),3) AS avg_compa_ratio_as_was,
         sum(case when f.below_band_at_event then 1 else 0 end) AS count_below_band_as_was
  FROM gold.fact_workforce_event f
  JOIN gold.dim_date d ON f.date_key = d.date_key
  WHERE f.base_salary_cad IS NOT NULL
  GROUP BY d.year ORDER BY d.year""").show()